In [3]:
# ==========================================
# GOOGLE COLAB NOTEBOOK VERSION
# ==========================================
# Run this entire script in a single Colab cell
# CSV file should be in Google Drive
# ==========================================

!pip install -q earthengine-api geemap pandas

import ee
import pandas as pd
import os

# ==========================================
# MOUNT GOOGLE DRIVE
# ==========================================

from google.colab import drive
drive.mount('/content/drive')
import time

# ==========================================
# AUTHENTICATE & INITIALIZE
# ==========================================

ee.Authenticate()
PROJECT_ID = "YOUR_GEE_PROJECT_ID"
ee.Initialize(project=PROJECT_ID)

# ==========================================
# LOAD TOP 100 CITIES FROM GOOGLE DRIVE
# ==========================================

# Update this path to match your Drive folder
DRIVE_FOLDER = "/content/drive/MyDrive/EE_Exports"
os.makedirs(DRIVE_FOLDER, exist_ok=True)

cities_df = pd.read_csv(f"{DRIVE_FOLDER}/india_top_cities_100.csv")
print(f"Loaded {len(cities_df)} cities")

# ==========================================
# CREATE GEE FEATURE COLLECTION FROM CITIES
# ==========================================

city_features = []
for idx, row in cities_df.iterrows():
    feature = ee.Feature(
        ee.Geometry.Point([row["lng"], row["lat"]]),
        {
            "City": str(row["city_ascii"]),
            "State": str(row["admin_name"]),
            "Latitude": float(row["lat"]),
            "Longitude": float(row["lng"]),
        }
    )
    city_features.append(feature)

cities_fc = ee.FeatureCollection(city_features)

# ==========================================
# DATASETS & CONFIG
# ==========================================

elevation = ee.Image("USGS/SRTMGL1_003").select("elevation").rename("Elevation")

months = {
    "January":   ("01-01", "01-31"),
    "February":  ("02-01", "02-28"),
    "March":     ("03-01", "03-31"),
    "April":     ("04-01", "04-30"),
    "May":       ("05-01", "05-31"),
    "June":      ("06-01", "06-30"),
    "July":      ("07-01", "07-31"),
    "August":    ("08-01", "08-31"),
    "September": ("09-01", "09-30"),
    "October":   ("10-01", "10-31"),
    "November":  ("11-01", "11-30"),
    "December":  ("12-01", "12-31"),
}

years = [2021, 2022, 2023, 2024, 2025]

os.makedirs("data", exist_ok=True)

# ==========================================
# PROCESS EACH YEAR
# ==========================================

for year in years:
    print(f"\n{'='*60}")
    print(f"  YEAR: {year}")
    print(f"{'='*60}")

    all_rows = []

    for month_name, (start_mm_dd, end_mm_dd) in months.items():
        start_date = f"{year}-{start_mm_dd}"
        end_date   = f"{year}-{end_mm_dd}"

        print(f"  {month_name}...", end=" ", flush=True)

        try:
            t0 = time.time()

            # Sentinel-2 median composite
            s2 = (
                ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED")
                .filterDate(start_date, end_date)
                .filter(ee.Filter.lt("CLOUDY_PIXEL_PERCENTAGE", 90))
                .median()
            )

            ndvi = s2.normalizedDifference(["B8", "B4"]).rename("NDVI")
            ndbi = s2.normalizedDifference(["B11", "B8"]).rename("NDBI")
            ndwi = s2.normalizedDifference(["B3", "B8"]).rename("NDWI")

            # MODIS Land Surface Temperature -> Celsius
            lst = (
                ee.ImageCollection("MODIS/061/MOD11A2")
                .filterDate(start_date, end_date)
                .select("LST_Day_1km")
                .mean()
                .multiply(0.02)
                .subtract(273.15)
                .rename("LST")
            )

            combined = ndvi.addBands(ndbi).addBands(ndwi).addBands(elevation).addBands(lst)

            # Sample ALL cities at once (batch)
            sampled = combined.sampleRegions(
                collection=cities_fc,
                scale=30,
                geometries=False
            )

            sampled = sampled.map(lambda f: f.set("Year", year).set("Month", month_name))

            results = sampled.getInfo()

            dt = time.time() - t0
            print(f"{len(results['features'])} cities in {dt:.1f}s")

            for feat in results["features"]:
                p = feat["properties"]
                all_rows.append({
                    "City":          p.get("City", ""),
                    "State":         p.get("State", ""),
                    "Latitude":      p.get("Latitude", ""),
                    "Longitude":     p.get("Longitude", ""),
                    "Year":          year,
                    "Month":         month_name,
                    "NDVI":          round(p["NDVI"], 4) if p.get("NDVI") is not None else None,
                    "NDBI":          round(p["NDBI"], 4) if p.get("NDBI") is not None else None,
                    "NDWI":          round(p["NDWI"], 4) if p.get("NDWI") is not None else None,
                    "Elevation (m)": p.get("Elevation", None),
                    "LST (°C)":      round(p["LST"], 2) if p.get("LST") is not None else None,
                })

        except Exception as e:
            print(f"Error: {e}")
            for _, cr in cities_df.iterrows():
                all_rows.append({
                    "City": cr["city_ascii"], "State": cr["admin_name"],
                    "Latitude": cr["lat"], "Longitude": cr["lng"],
                    "Year": year, "Month": month_name,
                    "NDVI": None, "NDBI": None, "NDWI": None,
                    "Elevation (m)": None, "LST (°C)": None,
                })

    # Save
    df = pd.DataFrame(all_rows)
    df = df[["City", "State", "Latitude", "Longitude", "Year", "Month",
             "NDVI", "NDBI", "NDWI", "Elevation (m)", "LST (°C)"]]

    path = f"{DRIVE_FOLDER}/india_cities_dataset_{year}.csv"
    df.to_csv(path, index=False)
    print(f"\n  Saved to Drive: {path} ({len(df)} rows)")

# ==========================================
# DONE
# ==========================================

print("\n" + "="*60)
print("  ALL DONE! Files saved to Google Drive.")
print("="*60)
print(f"\n Drive folder: {DRIVE_FOLDER}")
for year in years:
    print(f"  india_cities_dataset_{year}.csv  (1200 rows = 100 cities × 12 months)")


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Loaded 100 cities

  YEAR: 2021
  January... 99 cities in 2.0s
  February... 99 cities in 3.3s
  March... 99 cities in 6.0s
  April... 99 cities in 0.4s
  May... 98 cities in 3.5s
  June... 98 cities in 8.6s
  July... 83 cities in 0.3s
  August... 80 cities in 3.0s
  September... 72 cities in 3.7s
  October... 98 cities in 0.4s
  November... 96 cities in 3.1s
  December... 99 cities in 3.6s

  Saved to Drive: /content/drive/MyDrive/EE_Exports/india_cities_dataset_2021.csv (1120 rows)

  YEAR: 2022
  January... 99 cities in 0.7s
  February... 99 cities in 3.4s
  March... 99 cities in 3.8s
  April... 99 cities in 0.6s
  May... 99 cities in 5.3s
  June... 92 cities in 4.8s
  July... 73 cities in 0.4s
  August... 89 cities in 3.0s
  September... 88 cities in 3.8s
  October... 99 cities in 0.3s
  November... 99 cities in 3.2s
  December... 99 cities in 4.1s

  Sav

In [4]:
# ==========================================
# MERGE ALL 5 YEARLY CSVs INTO ONE
# ==========================================
# Run this in a separate Colab cell AFTER
# the main script has finished saving all files
# ==========================================

import pandas as pd

from google.colab import drive
drive.mount('/content/drive')

# ==========================================
# FETCH CSVs FROM GOOGLE DRIVE
# ==========================================

DRIVE_FOLDER = "/content/drive/MyDrive/EE_Exports"
years = [2021, 2022, 2023, 2024, 2025]

all_dfs = []
for year in years:
    path = f"{DRIVE_FOLDER}/india_cities_dataset_{year}.csv"
    df = pd.read_csv(path)
    all_dfs.append(df)
    print(f"  Loaded: india_cities_dataset_{year}.csv  ({len(df)} rows)")

# ==========================================
# MERGE & SAVE
# ==========================================

merged_df = pd.concat(all_dfs, ignore_index=True)

merged_path = f"{DRIVE_FOLDER}/india_cities_dataset_2021_2025_merged.csv"
merged_df.to_csv(merged_path, index=False)

print(f"\n Merged & saved to Drive!")
print(f" {merged_path}")
print(f"   Total rows: {len(merged_df)} (100 cities × 12 months × 5 years)")
print(f"\nPreview:")
print(merged_df.head())


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
  Loaded: india_cities_dataset_2021.csv  (1120 rows)
  Loaded: india_cities_dataset_2022.csv  (1134 rows)
  Loaded: india_cities_dataset_2023.csv  (1131 rows)
  Loaded: india_cities_dataset_2024.csv  (1084 rows)
  Loaded: india_cities_dataset_2025.csv  (1118 rows)

 Merged & saved to Drive!
 /content/drive/MyDrive/EE_Exports/india_cities_dataset_2021_2025_merged.csv
   Total rows: 5587 (100 cities × 12 months × 5 years)

Preview:
        City        State  Latitude  Longitude  Year    Month    NDVI    NDBI  \
0     Mumbai  Mahārāshtra   19.0758    72.8775  2021  January  0.0371  0.0316   
1    Kolkata  West Bengal   22.5675    88.3700  2021  January  0.0715  0.0436   
2      Delhi        Delhi   28.6600    77.2300  2021  January  0.0814 -0.0198   
3    Chennai   Tamil Nādu   13.0825    80.2750  2021  January  0.0603  0.1507   
4  Bangalore    Karnātaka   12.9